# Ours vs. the City's APR — A Reconciliation

This notebook compares **our independent CO count** (from `berkeley_housing_v2.db`, computed in
*Notebook 1*) against the **city's submitted APR** (mirrored from the state CKAN portal), project by
project. The goal is honest reconciliation: **broad agreement**, a small set of **verified differences**,
and an explicit account of every divergence as *genuine error*, *convention difference*, or *scope*.

**Discipline: verified-before-characterized.** Every difference below was checked against the primary
permit record (CPRA Finaled dates) before being named. An earlier draft of this analysis asserted several
"city errors" that turned out to be **bugs in our own comparison queries** — those are documented in the
Methodology Corrections section so the reader can see what was retracted and why.

_Read-only: queries both databases, modifies neither._

## Setup — connect both sources read-only

In [1]:
import sqlite3, hashlib, re
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
while not (ROOT/'databases'/'berkeley_housing_v2.db').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
V2  = ROOT/'databases'/'berkeley_housing_v2.db'
HCD = ROOT/'databases'/'hcd_apr_mirror.db'   # the city's submitted APR, mirrored from CKAN
print('v2 canonical SHA:', hashlib.sha256(V2.read_bytes()).hexdigest()[:12])
v   = sqlite3.connect(f'file:{V2}?mode=ro', uri=True)
hcd = sqlite3.connect(f'file:{HCD}?mode=ro', uri=True)
UC  = (165,170,171,177)
def napn(a): return re.sub(r'[^0-9]','', str(a or ''))

v2 canonical SHA: 02f3cfa9207c


## Method & a guardrail learned the hard way

Two query rules, both the result of bugs we caught:

1. **Sum only the income/unit columns — never the date columns.** HCD's `table_a2` interleaves unit
   columns (`CO_*_INCOME`) with date columns (`CO_ISSUE_DT1`). An earlier query summed *all* `CO_`/`BP_`
   columns; SQLite coerced a date string like `'2021-10-27'` to the integer `2021` and added it to the
   real unit count — producing phantom values like **2190** (= 169 real units + 2021 date) for a
   169-unit project. We now sum the income columns explicitly.
2. **Match by APN with a sanity gate.** Address `LIKE` matching missed rows (`'2150 KITTREDGE St'` vs
   `'2150-76 Kittredge'`). We match on normalized APN, and **any single project over ~600 units, or a
   year total wildly above ours, trips a 'verify against source' flag** before interpretation.

In [2]:
cols = [c[1] for c in hcd.execute("PRAGMA table_info(table_a2)")]
CO_UNIT = [c for c in cols if c.startswith('CO_') and 'INCOME' in c and 'DT' not in c]
SANITY = 600
def co_units(row):
    u = sum(int(float(row[c])) if str(row[c]).strip() not in ('','None') else 0 for c in CO_UNIT)
    return u
# validate the guardrail: 2150 Kittredge MUST be 169, not 2190
hcd.row_factory = sqlite3.Row
chk = [co_units(r) for r in hcd.execute("SELECT * FROM table_a2 WHERE JURIS_NAME='BERKELEY' AND APN='057 202901600'")]
print('guardrail check — 2150 Kittredge CO units per row:', chk, '(must be [0, 169], never 2190)')
assert 2190 not in chk, 'date-column bug has returned'

guardrail check — 2150 Kittredge CO units per row: [0, 169] (must be [0, 169], never 2190)


## Per-year: the city's APR vs ours, 2018-2026

In [3]:
def city_year(y):
    seen, tripped = {}, []
    for r in hcd.execute(f"SELECT * FROM table_a2 WHERE YEAR={y} AND JURIS_NAME='BERKELEY'"):
        u = co_units(r)
        if u <= 0: continue
        if u > SANITY: tripped.append((r['STREET_ADDRESS'], u))   # sanity gate
        k = napn(r['APN']) or 'X'+str(r['STREET_ADDRESS'])
        seen[k] = max(seen.get(k,0), u)
    return sum(seen.values()), len(seen), tripped
rows = []
for y in range(2018, 2027):
    cu, cp, trip = city_year(y)
    ou = v.execute('SELECT COALESCE(SUM(total_units),0) FROM v_projects_flat '
                   f"WHERE substr(co_issued_date,1,4)='{y}' AND project_id NOT IN {UC}").fetchone()[0]
    assert not trip, f'SANITY TRIP in {y}: {trip} — verify against source'
    rows.append({'Year': y, 'City APR (CKAN)': cu, 'Ours (v2)': ou, 'Diff (ours-city)': ou-cu})
recon = pd.DataFrame(rows).set_index('Year')
display(recon)

,City APR (CKAN),Ours (v2),Diff (ours-city)
Year,,,
2018,226,70,-156
2019,307,98,-209
2020,399,76,-323
2021,323,107,-216
2022,826,84,-742
2023,704,700,-4
2024,706,709,3
2025,487,531,44
2026,0,216,216


## Reading the table

- **2018–2022 — scope, not disagreement.** Our v2 currently holds only the **ADU/small** backfill for
  these pre-policy years; the pre-policy *major* projects are a separate, later verification pass. So the
  large city-higher gaps here are **our incomplete coverage**, not city error. (See Notebook 1's ADU/MF
  split — 2018–2022 are 100% ADU.)
- **2023–2025 — the meaningful comparison, and it is close.** This is where v2 is complete (ADU + majors):
  - **CY2023: ours 700 vs city 704 (−4).** After adding the Logan Park **South Building** (a genuine
    2023 completion we had been missing — found during this reconciliation), the remaining 4u is **1u
    (2210 MLK, a held year-disagreement) + ~3u (Rule-C net-new-vs-reported convention).**
  - **CY2024: ours 709 vs city 706 (+3).** Rule-C convention (we count net-new slightly differently).
  - **CY2025: ours 531 vs city 487 (+44).** We are **ahead** — comprehensive ADU coverage the city's
    APR omitted, plus **1367 University** (39u): the city carries it as a building permit only (2023),
    while we count its completion (permit B2022-04366, **Finaled 2025-05-06**, primary-confirmed).
- **CY2026:** the city's mirror predates the 2026 APR, so it shows 0; our 216 is this year's completions.

## The surviving differences, named (each verified against the permit record)

In [4]:
findings = pd.DataFrame([
  {'Year':2023, 'Item':'2210 MLK', 'Units':1, 'Type':'genuine year-disagreement (HELD)',
   'Detail':'City CO 2023-01-11; our CPRA-backed v2 date 2025-03-26. Unresolved — needs legacy permit lookup.'},
  {'Year':2023, 'Item':'Rule-C delta', 'Units':3, 'Type':'convention',
   'Detail':'Our net-new (Rule C) vs the city reported figure; the documented 441-vs-444 class.'},
  {'Year':2024, 'Item':'Rule-C delta', 'Units':3, 'Type':'convention',
   'Detail':'We count 3u more net-new than the city across shared projects.'},
  {'Year':2025, 'Item':'1367 University', 'Units':39, 'Type':'we caught a completion the city omitted',
   'Detail':'City: BP-only (2023, no CO). Us: completed, permit B2022-04366 Finaled 2025-05-06 (primary-confirmed).'},
  {'Year':2025, 'Item':'comprehensive ADU + Rule-C', 'Units':5, 'Type':'coverage + convention',
   'Detail':'Small ADU/unit-count differences on the ADU set vs the city APR.'},
])
display(findings)

,Year,Item,Units,Type,Detail
0,2023,2210 MLK,1,genuine year-disagreement (HELD),City CO 2023-01-11; our CPRA-backed v2 date 20...
1,2023,Rule-C delta,3,convention,Our net-new (Rule C) vs the city reported figu...
2,2024,Rule-C delta,3,convention,We count 3u more net-new than the city across ...
3,2025,1367 University,39,we caught a completion the city omitted,"City: BP-only (2023, no CO). Us: completed, pe..."
4,2025,comprehensive ADU + Rule-C,5,coverage + convention,Small ADU/unit-count differences on the ADU se...


## Methodology corrections — findings that did NOT survive verification (retracted)

An earlier draft asserted several **"premature-completion city errors"**. Checked against the primary
permit record, **every one dissolved into agreement** — they were artifacts of the two query bugs above:

| Claimed | Reality (verified) |
|---|---|
| 2150 Kittredge — "city counted prematurely (2021), 2190-unit error" | HCD shows **169u, CO 2024-03-20** — agrees with us; "2190" was the date-sum bug; the 2024 row was missed by a `LIKE` bug |
| 2480 Bancroft / 2440 Shattuck / 2555 College — "city counted before Finaled" | HCD's CO year/units/date **match ours exactly** (2025, to the day) |
| Logan Park — "CKAN mis-yeared to 2023" | The North Building **is** 2022 (agree); a **separate South Building** genuinely finaled 2023-08-08 — *we* had been missing it, now ingested. **We under-counted; the city was right.** |
| 2029 University — "city CO double-count (240+160)" | No CO for it in either source (Entitled). Not a completion finding. |

**What stands:** the **Accela/CPRA-sourced Finaled dates** (read directly from permits) were never
affected by the SQL bug and remain valid. Only the *CKAN-comparison conclusions* needed re-deriving, and
most became agreement.

### Provenance flag
City unit counts quoted in any comparison are **city-derived** (HCD's published figures); our counts are
**primary-permit-confirmed** (CPRA Finaled records + Alameda assessor). Where the two differ, the permit
record governs — that is what turned four "city errors" into agreement and one of *our* gaps into a real
completion we had missed.

---
*Bottom line: broad agreement with the city's APR; scope-limited pre-policy years; and a handful of
verified, named differences — no unexamined "city error" claims survive in this notebook.*

In [5]:
v.close(); hcd.close()